In [1]:
from pathlib import Path
import csv

pasta_atual = Path.cwd()
raiz = pasta_atual.parent if pasta_atual.name == "notebooks" else pasta_atual

pasta_processados = raiz / "data" / "processed"

def carregar_csv(nome):
    caminho = pasta_processados / nome

    with caminho.open("r", encoding="utf-8-sig", newline="") as arquivo:
        return list(csv.DictReader(arquivo, delimiter=";"))

registros_pdf = carregar_csv("textos_pdfs.csv")
registros_html = carregar_csv("textos_html.csv")

print("Raiz utilizada:", raiz)
print("Registros PDF:", len(registros_pdf))
print("Registros HTML:", len(registros_html))
print("Arquivos PDF distintos:", len({r["arquivo"] for r in registros_pdf}))
print("Arquivos HTML distintos:", len({r["arquivo"] for r in registros_html}))

Raiz utilizada: /Users/hugofilho/Documents/projeto-pln
Registros PDF: 116
Registros HTML: 14
Arquivos PDF distintos: 4
Arquivos HTML distintos: 14


In [2]:
from collections import defaultdict

resumo_fontes = defaultdict(
    lambda: {"registros": 0, "caracteres": 0, "sem_texto": 0}
)

for registro in registros_pdf + registros_html:
    arquivo = registro["arquivo"]
    texto = registro["texto"]

    resumo_fontes[arquivo]["registros"] += 1
    resumo_fontes[arquivo]["caracteres"] += len(texto.strip())
    resumo_fontes[arquivo]["sem_texto"] += int(not texto.strip())

for arquivo, resumo in sorted(resumo_fontes.items()):
    print(
        f"{arquivo}\n"
        f"  Registros: {resumo['registros']} | "
        f"Caracteres: {resumo['caracteres']} | "
        f"Sem texto: {resumo['sem_texto']}\n"
    )

05.html
  Registros: 1 | Caracteres: 5850 | Sem texto: 0

06.html
  Registros: 1 | Caracteres: 5503 | Sem texto: 0

07.html
  Registros: 1 | Caracteres: 2468 | Sem texto: 0

08.html
  Registros: 1 | Caracteres: 3050 | Sem texto: 0

09.html
  Registros: 1 | Caracteres: 1427 | Sem texto: 0

10.html
  Registros: 1 | Caracteres: 1093 | Sem texto: 0

11.html
  Registros: 1 | Caracteres: 1292 | Sem texto: 0

12.html
  Registros: 1 | Caracteres: 8758 | Sem texto: 0

13.html
  Registros: 1 | Caracteres: 2601 | Sem texto: 0

14.html
  Registros: 1 | Caracteres: 5000 | Sem texto: 0

15.html
  Registros: 1 | Caracteres: 9704 | Sem texto: 0

16.html
  Registros: 1 | Caracteres: 2342 | Sem texto: 0

17.html
  Registros: 1 | Caracteres: 1930 | Sem texto: 0

18.html
  Registros: 1 | Caracteres: 704 | Sem texto: 0

Calendario_Academico_2026_2_Ibmec_SP_v5_08310586a3.pdf
  Registros: 6 | Caracteres: 3325 | Sem texto: 0

GradManualAlunoIbmec.pdf
  Registros: 49 | Caracteres: 77971 | Sem texto: 1

GuiaCal

In [3]:
assert len(resumo_fontes) == 18, "Esperávamos 18 fontes."

revisao_fontes = {
    arquivo: {
        "decisao": "pendente",
        "observacao": "",
    }
    for arquivo in resumo_fontes
}

observacoes_iniciais = {
    "06.html": "Conferir atualidade das instruções de acesso e senha.",
    "12.html": "Distinguir serviços por público e unidade.",
    "14.html": "Separar orientações gerais dos prazos encerrados de 2026.1.",
    "15.html": "Distinguir divulgação do curso de regras acadêmicas.",
    "16.html": "Avaliar como índice para localizar documentos.",
}

for arquivo, observacao in observacoes_iniciais.items():
    revisao_fontes[arquivo]["observacao"] = observacao

print("Fontes registradas para revisão:", len(revisao_fontes))

for arquivo, revisao in sorted(revisao_fontes.items()):
    if revisao["observacao"]:
        print(f"\n{arquivo}: {revisao['observacao']}")

Fontes registradas para revisão: 18

06.html: Conferir atualidade das instruções de acesso e senha.

12.html: Distinguir serviços por público e unidade.

14.html: Separar orientações gerais dos prazos encerrados de 2026.1.

15.html: Distinguir divulgação do curso de regras acadêmicas.

16.html: Avaliar como índice para localizar documentos.


In [4]:
import re

registro_colacao = next(
    registro
    for registro in registros_html
    if registro["arquivo"] == "14.html"
)

texto_colacao = registro_colacao["texto"]

datas_colacao = list(
    re.finditer(r"\b\d{2}/\d{2}/\d{4}\b", texto_colacao)
)

print("Datas encontradas:", len(datas_colacao))

for ocorrencia in datas_colacao:
    inicio = max(0, ocorrencia.start() - 180)
    fim = min(len(texto_colacao), ocorrencia.end() + 220)

    print(f"\nDATA: {ocorrencia.group()}")
    print(texto_colacao[inicio:fim])

Datas encontradas: 2

DATA: 25/05/2026
egularizar em "Escolher arquivo"  e adicionar o arquivo correspondente, em seguida, é só clicar em "Submeter" e acompanhar a análise da documentação. Simples, né? 😊 ❗ Você tem até 25/05/2026 para regularizar sua documentação junto à instituição, para que possa realizar a colação de grau. 🎓: "E sobre as Atividades Acadêmicas Complementar (AAC)?" 💙: As Atividades Acadêmicas Complementares ou AAC são obrigatór

DATA: 01/06/2026
horas para colar grau e receber o diploma. São dois tipos de AAC: Internas e Externas.
Para consultar suas horas, acesse o SIA; Atividades Acadêmicas Complementares ❗ Você tem até 01/06/2026 para cumprir o mínimo de horas AAC exigidas, conforme matriz curricular do seu curso, para que possa realizar a colação de grau. É Importante ficar atento ao período para solicitar o cadastro das horas de AAC Externas, 


In [5]:
revisao_fontes["14.html"].update({
    "decisao": "uso_parcial_pendente",
    "observacao": (
        "Contém prazos encerrados: documentação em 25/05/2026 "
        "e AAC em 01/06/2026. Separar os trechos históricos antes "
        "de usar na busca. Orientações gerais ainda exigem revisão "
        "de atualidade e abrangência."
    ),
})

print(revisao_fontes["14.html"]["decisao"])
print(revisao_fontes["14.html"]["observacao"])

uso_parcial_pendente
Contém prazos encerrados: documentação em 25/05/2026 e AAC em 01/06/2026. Separar os trechos históricos antes de usar na busca. Orientações gerais ainda exigem revisão de atualidade e abrangência.


In [6]:
termos = [
    "senha padrão",
    "6 caracteres",
    "primeiro acesso",
    "cadastrar primeira senha",
]

for nome in ["05.html", "06.html"]:
    registro = next(
        r for r in registros_html if r["arquivo"] == nome
    )
    texto = registro["texto"]

    intervalos = []

    for termo in termos:
        for ocorrencia in re.finditer(re.escape(termo), texto, re.IGNORECASE):
            intervalos.append((
                max(0, ocorrencia.start() - 150),
                min(len(texto), ocorrencia.end() + 300),
            ))

    # Une intervalos sobrepostos para não repetir o mesmo trecho.
    trechos = []

    for inicio, fim in sorted(intervalos):
        if trechos and inicio <= trechos[-1][1]:
            trechos[-1][1] = max(trechos[-1][1], fim)
        else:
            trechos.append([inicio, fim])

    print(f"\n--- {nome} ---")

    for inicio, fim in trechos:
        print(texto[inicio:fim])
        print()


--- 05.html ---
 de Informações Acadêmicas)?

Acesse o Portal SIA e faça seu login utilizando seu e-mail de estudante e senha ou usando o número da sua matrícula. No primeiro acesso, você pode consultar seu número de matrícula clicando na opção “Não sei ou esqueci a matrícula” e criar sua senha clicando na opção “Esqueci minha senha / Cadastrar primeira senha”;

No SIA, você tem acesso à todas as informações que precisa, além dos ambientes onlines do Ibmec, como a Sala de Aula Virtual. Ela é uma ferramenta de apoio para suas atividades acadêmicas, oferecendo materiais e conteúdos complementares. Você pode acessá-la através do menu no SIA, no App do Alun


--- 06.html ---
01. Como realizar meu primeiro acesso?
No SIA, é possível realizar o primeiro acesso de duas maneiras: via e-mail de estudante e via matrícula.
Confira abaixo os dois formatos de Acesso:
1.1 Acesso via e-mail de estudante:
Para realizar seu primeiro acesso via e-mail de estudante, você precisa utilizar sua conta Micro

In [7]:
revisao_fontes["06.html"].update({
    "decisao": "uso_parcial_pendente",
    "observacao": (
        "Distingue acesso por e-mail institucional e por matrícula. "
        "Atualidade da senha padrão e dos requisitos de senha não confirmada. "
        "Manter esses trechos fora da busca até validação. "
        "Revisar as demais orientações considerando o método de acesso."
    ),
})

print(revisao_fontes["06.html"]["observacao"])

Distingue acesso por e-mail institucional e por matrícula. Atualidade da senha padrão e dos requisitos de senha não confirmada. Manter esses trechos fora da busca até validação. Revisar as demais orientações considerando o método de acesso.


In [8]:
decisoes = {
    "12.html": {
        "decisao": "uso_parcial_pendente",
        "observacao": (
            "Selecionar serviços aplicáveis à graduação. "
            "Preservar o público e a unidade associados a cada serviço. "
            "Contatos de São Paulo não especificam Faria Lima."
        ),
    },
    "15.html": {
        "decisao": "referencia_pendente",
        "observacao": (
            "Página de divulgação com dados incorporados sobre o curso. "
            "Não usar como comprovação de regras ou currículo sem verificar "
            "unidade, modalidade e versão."
        ),
    },
    "16.html": {
        "decisao": "localizar_documentos",
        "observacao": (
            "Índice para localizar documentos oficiais por unidade. "
            "Os documentos apontados precisam ser examinados separadamente."
        ),
    },
}

for arquivo, decisao in decisoes.items():
    revisao_fontes[arquivo].update(decisao)
    print(f"{arquivo}: {revisao_fontes[arquivo]['decisao']}")

12.html: uso_parcial_pendente
15.html: referencia_pendente
16.html: localizar_documentos


In [9]:
from collections import defaultdict
import re

grupos_textos = defaultdict(list)

for registro in registros_pdf + registros_html:
    texto_normalizado = re.sub(
        r"\s+", " ", registro["texto"]
    ).strip().casefold()

    if not texto_normalizado:
        continue

    grupos_textos[texto_normalizado].append({
        "arquivo": registro["arquivo"],
        "pagina": registro.get("pagina", ""),
        "mes": registro.get("mes", ""),
    })

duplicatas_exatas = [
    origens
    for origens in grupos_textos.values()
    if len(origens) > 1
]

print("Grupos com texto duplicado:", len(duplicatas_exatas))

for grupo in duplicatas_exatas:
    print("\nMesmo texto encontrado em:")
    for origem in grupo:
        print(origem)

Grupos com texto duplicado: 0


In [10]:
abrangencia_pdfs = {
    "GradManualAlunoIbmec.pdf": (
        "Manual geral de graduação; verificar exceções por unidade "
        "e confirmar versão."
    ),
    "GuiadoAlunoIbmecFariaLima.pdf": (
        "Identificado como Faria Lima; confirmar versão e abrangência "
        "das regras apresentadas."
    ),
    "GuiaCalouroIbmecFL.pdf": (
        "Identificado como Faria Lima; revisar atualidade dos contatos "
        "e orientações."
    ),
    "Calendario_Academico_2026_2_Ibmec_SP_v5_08310586a3.pdf": (
        "São Paulo, semestre 2026.2; recortes mensais conferidos. "
        "Preservar exceções específicas da unidade Paulista."
    ),
}

for arquivo, observacao in abrangencia_pdfs.items():
    revisao_fontes[arquivo].update({
        "decisao": "candidato_pendente",
        "observacao": observacao,
    })

print("PDFs registrados:", len(abrangencia_pdfs))
print("Grupos de duplicatas exatas:", len(duplicatas_exatas))

PDFs registrados: 4
Grupos de duplicatas exatas: 0


In [11]:
linhas_revisao = [
    {
        "arquivo": arquivo,
        "decisao": revisao["decisao"],
        "observacao": revisao["observacao"],
    }
    for arquivo, revisao in sorted(revisao_fontes.items())
]

destino_revisao = pasta_processados / "revisao_fontes.csv"

with destino_revisao.open(
    "x", encoding="utf-8-sig", newline=""
) as arquivo:
    escritor = csv.DictWriter(
        arquivo,
        fieldnames=["arquivo", "decisao", "observacao"],
        delimiter=";",
    )
    escritor.writeheader()
    escritor.writerows(linhas_revisao)

with destino_revisao.open(
    "r", encoding="utf-8-sig", newline=""
) as arquivo:
    revisao_recuperada = list(
        csv.DictReader(arquivo, delimiter=";")
    )

assert revisao_recuperada == linhas_revisao, (
    "A revisão recuperada difere dos dados originais."
)

print("Arquivo salvo:", destino_revisao)
print("Fontes registradas:", len(revisao_recuperada))
print("Gravação conferida.")

Arquivo salvo: /Users/hugofilho/Documents/projeto-pln/data/processed/revisao_fontes.csv
Fontes registradas: 18
Gravação conferida.


In [12]:
fora_da_versao_inicial = {
    "06.html": "Instruções de senha com atualidade não confirmada.",
    "12.html": "Separação por público e unidade ainda não implementada.",
    "13.html": "Confirmar o período de renovação ao qual o texto se refere.",
    "14.html": "Trechos com prazos encerrados ainda não foram separados.",
    "15.html": "Fora do escopo inicial de orientações e serviços acadêmicos.",
    "16.html": "Usar para localizar documentos, não como fonte de regras.",
}

for arquivo, revisao in revisao_fontes.items():
    if arquivo in fora_da_versao_inicial:
        revisao["decisao"] = "fora_da_versao_inicial"
        revisao["observacao"] = fora_da_versao_inicial[arquivo]
    else:
        revisao["decisao"] = "selecionada_base_experimental"

fontes_selecionadas = {
    arquivo
    for arquivo, revisao in revisao_fontes.items()
    if revisao["decisao"] == "selecionada_base_experimental"
}

print("Fontes selecionadas:", len(fontes_selecionadas))
print("Fontes fora da versão inicial:", len(fora_da_versao_inicial))

Fontes selecionadas: 12
Fontes fora da versão inicial: 6


In [13]:
corpus_inicial = []
registros_excluidos = []

for tipo, registros in [
    ("pdf", registros_pdf),
    ("html", registros_html),
]:
    for registro in registros:
        arquivo = registro["arquivo"]

        if arquivo not in fontes_selecionadas:
            motivo = "Fonte fora da versão inicial."
        elif not registro["texto"].strip():
            motivo = "Registro sem texto extraível."
        else:
            motivo = None

        if motivo:
            registros_excluidos.append({
                "arquivo": arquivo,
                "pagina": registro.get("pagina", ""),
                "mes": registro.get("mes", ""),
                "motivo": motivo,
            })
            continue

        copia = registro.copy()
        copia["tipo"] = tipo
        copia["status_revisao"] = "base_experimental_revisao_amostral"
        copia["observacao_fonte"] = revisao_fontes[arquivo]["observacao"]

        corpus_inicial.append(copia)

print("Registros selecionados:", len(corpus_inicial))
print("Registros excluídos:", len(registros_excluidos))
print("Fontes selecionadas:", len({r["arquivo"] for r in corpus_inicial}))

Registros selecionados: 122
Registros excluídos: 8
Fontes selecionadas: 12


In [14]:
colunas_corpus = [
    "tipo", "arquivo", "titulo", "url", "pagina", "mes",
    "ano", "semestre", "texto", "status_revisao",
    "observacao", "observacao_fonte",
]

linhas_corpus = [
    {coluna: str(registro.get(coluna, "")) for coluna in colunas_corpus}
    for registro in corpus_inicial
]

linhas_revisao_final = [
    {
        "arquivo": arquivo,
        "decisao": revisao["decisao"],
        "observacao": revisao["observacao"],
    }
    for arquivo, revisao in sorted(revisao_fontes.items())
]

saidas = {
    "corpus_inicial.csv": linhas_corpus,
    "exclusoes_corpus.csv": registros_excluidos,
    "revisao_fontes_v1.csv": linhas_revisao_final,
}

# Confere todos os destinos antes de começar a gravação.
for nome in saidas:
    if (pasta_processados / nome).exists():
        raise FileExistsError(f"{nome} já existe; nada foi sobrescrito.")

for nome, linhas in saidas.items():
    caminho = pasta_processados / nome
    colunas = list(linhas[0])

    with caminho.open("x", encoding="utf-8-sig", newline="") as arquivo:
        escritor = csv.DictWriter(
            arquivo, fieldnames=colunas, delimiter=";"
        )
        escritor.writeheader()
        escritor.writerows(linhas)

    with caminho.open("r", encoding="utf-8-sig", newline="") as arquivo:
        recuperadas = list(csv.DictReader(arquivo, delimiter=";"))

    esperadas = [
        {chave: str(valor) for chave, valor in linha.items()}
        for linha in linhas
    ]

    assert recuperadas == esperadas, f"Falha na conferência de {nome}"
    print(f"{nome}: {len(recuperadas)} registros — gravação conferida")

corpus_inicial.csv: 122 registros — gravação conferida
exclusoes_corpus.csv: 8 registros — gravação conferida
revisao_fontes_v1.csv: 18 registros — gravação conferida


In [15]:
corpus_salvo = carregar_csv("corpus_inicial.csv")
exclusoes_salvas = carregar_csv("exclusoes_corpus.csv")
revisao_salva = carregar_csv("revisao_fontes_v1.csv")

originais = (
    carregar_csv("textos_pdfs.csv")
    + carregar_csv("textos_html.csv")
)

def chave(registro):
    return (
        registro["arquivo"],
        registro.get("pagina", ""),
        registro.get("mes", ""),
    )

originais_por_chave = {chave(r): r for r in originais}
chaves_corpus = {chave(r) for r in corpus_salvo}
chaves_exclusoes = {chave(r) for r in exclusoes_salvas}

assert len(originais_por_chave) == len(originais)
assert len(corpus_salvo) == len(chaves_corpus) == 122
assert len(exclusoes_salvas) == len(chaves_exclusoes) == 8

# Nenhum registro pode estar simultaneamente incluído e excluído.
assert chaves_corpus.isdisjoint(chaves_exclusoes)

# Todos os registros originais precisam ter um destino.
assert chaves_corpus | chaves_exclusoes == set(originais_por_chave)

assert all(r["texto"].strip() for r in corpus_salvo)
assert all(
    r["texto"] == originais_por_chave[chave(r)]["texto"]
    for r in corpus_salvo
)

fontes_aprovadas_para_experimento = {
    r["arquivo"]
    for r in revisao_salva
    if r["decisao"] == "selecionada_base_experimental"
}

assert len(revisao_salva) == 18
assert len(fontes_aprovadas_para_experimento) == 12
assert {r["arquivo"] for r in corpus_salvo} == fontes_aprovadas_para_experimento

print("Conferência final concluída:")
print("122 registros selecionados, de 12 fontes.")
print("8 exclusões registradas.")
print("Todos os registros originais contabilizados.")
print("Textos preservados, sem registros vazios no corpus.")
print("Seleção consistente com a revisão das 18 fontes.")

Conferência final concluída:
122 registros selecionados, de 12 fontes.
8 exclusões registradas.
Todos os registros originais contabilizados.
Textos preservados, sem registros vazios no corpus.
Seleção consistente com a revisão das 18 fontes.
